# Microsoft Agent Framework — Azure OpenAI (Responses API)

In dit codevoorbeeld ga je de **Microsoft Agent Framework (MAF)** gebruiken om een eenvoudige agent te maken die wordt ondersteund door **Azure OpenAI** met behulp van de **Responses API**.

> **Migratienotitie:** Deze voorbeeldcode gebruikte eerder Semantic Kernel met GitHub-modellen. Het is gemigreerd naar het Microsoft Agent Framework, en GitHub-modellen (verouderd, gestopt in juli 2026) is vervangen door Azure OpenAI, dat de Responses API ondersteunt. De `OpenAIChatClient` in MAF richt zich op de stabiele `/openai/v1/` endpoint van Azure OpenAI en gebruikt standaard de Responses API.

Het doel van dit voorbeeld is om de stappen te demonstreren die later zullen worden toegepast in aanvullende codevoorbeelden bij het implementeren van verschillende agentpatronen.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Importeer de benodigde Python-pakketten


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Een Tool Definiëren

In het Microsoft Agent Framework is een **tool** een gewone Python-functie die is gedecoreerd met `@tool` en die de agent kan aanroepen. Hieronder definiëren we een tool die een willekeurige vakantiebestemming retourneert en voorkomt dat dezelfde bestemming als de vorige wordt herhaald.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## De agent maken

Hier maken we de agent genaamd `TravelAgent`.

In dit voorbeeld gebruiken we heel eenvoudige instructies. Voel je vrij om deze instructies aan te passen om te zien hoe het gedrag van de agent verandert.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## De agent uitvoeren

Nu kunnen we de agent uitvoeren. We maken een `AgentSession` aan zodat de agent het gesprek over beurten heen onthoudt, en verzenden vervolgens twee `user_inputs`. De eerste vraagt om een reis; de tweede zegt dat de gebruiker de suggestie niet leuk vond en vraagt om een andere — de agent gebruikt de sessiegeschiedenis plus de `get_random_destination` tool om te reageren.

Je kunt deze berichten aanpassen om te zien hoe de agent anders reageert. Reacties worden **gestreamd** token-voor-token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
